# Лаб. 2, варіант 2 
$y'' - e^{x}y' + y\cos x = x,\quad x\in[0,\pi],\quad y(0)=1,\ y(\pi)=0$

Центральні різниці, $h=\pi/N$:
$$\Big(1-\tfrac{h p_i}{2}\Big)y_{i-1} + \big(-2+h^2 q_i\big)y_i + \Big(1+\tfrac{h p_i}{2}\Big)y_{i+1} = h^2 f_i,\qquad p=-e^x,\ q=\cos x,\ f=x$$

Прогонка: $y_i = \alpha_i y_{i+1} + \beta_i$, $\ \alpha_i = \dfrac{-c_i}{b_i + a_i\alpha_{i-1}}$, $\ \beta_i = \dfrac{d_i - a_i\beta_{i-1}}{b_i + a_i\alpha_{i-1}}$



In [ ]:
import numpy as np; import matplotlib.pyplot as plt

def thomas(a, b, c, d):
    # a - below diagonal, b - diagonal, c - above; a[0] and c[-1] not used
    n = len(b)
    alpha = np.zeros(n)
    beta = np.zeros(n)
    alpha[0] = -c[0] / b[0]
    beta[0] = d[0] / b[0]
    for i in range(1, n):
        denom = b[i] + a[i] * alpha[i-1]
        alpha[i] = -c[i] / denom
        beta[i] = (d[i] - a[i] * beta[i-1]) / denom

    # backward pass
    x = np.zeros(n)
    x[-1] = beta[-1]
    for i in range(n - 2, -1, -1):
        x[i] = alpha[i] * x[i+1] + beta[i]
    return x

# quick sanity check vs numpy
n = 6
a = np.random.rand(n); c = np.random.rand(n); a[0] = 0; c[-1] = 0
b = a + c + 1 # diagonal dominance
d = np.random.rand(n)
A = np.diag(b) + np.diag(a[1:], -1) + np.diag(c[:-1], 1)
print(np.max(np.abs(thomas(a, b, c, d) - np.linalg.solve(A, d))))

: 

In [ ]:
p = lambda x: -np.exp(x)
q = lambda x: np.cos(x)
f = lambda x: x
x0, x1 = 0, np.pi
y0, y1 = 1, 0

def solve(N):
    x = np.linspace(x0, x1, N + 1)
    h = (x1 - x0) / N
    xi = x[1:-1] # unknowns are only interior points, y0 and yN are known

    a = 1 - h * p(xi) / 2
    b = -2 + h**2 * q(xi)
    c = 1 + h * p(xi) / 2
    d = h**2 * f(xi)

    # move boundary values to rhs
    d[0] -= a[0] * y0
    d[-1] -= c[-1] * y1

    return x, np.concatenate(([y0], thomas(a, b, c, d), [y1]))

# NOTE: |p| goes up to e^pi ~ 23, so need h*|p|/2 < 1  ->  h < 2/e^pi ~ 0.086 (N >= 37)
# otherwise c_i < 0, no diagonal dominance, solution oscillates
for N in [10, 20, 40, 100, 1000]:
    x, y = solve(N)
    plt.plot(x, y, '--' if N < 37 else '-', marker = 'o' if N <= 40 else None, markersize = 3, label = f'N = {N}')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
from scipy.integrate import solve_bvp

# check: scipy on the same problem as a 1st order system (u0 = y, u1 = y')
xs = np.linspace(x0, x1, 200)
sol = solve_bvp(lambda x, u: np.vstack([u[1], x + np.exp(x) * u[1] - np.cos(x) * u[0]]),
                lambda ua, ub: np.array([ua[0] - y0, ub[0] - y1]),
                xs, np.zeros((2, xs.size)), tol = 1e-10, max_nodes = 100000)

# error should drop ~4x when N doubles (scheme is O(h^2))
for N in [50, 100, 200, 400, 800]:
    x, y = solve(N)
    print(N, np.max(np.abs(y - sol.sol(x)[0])))